# Install Dependencies

This cell installs the required packages for YOLOv8 training and YAML parsing.

In [7]:
!pip install ultralytics pyyaml

# Configuration

Centralized model and dataset configuration values used throughout the notebook.


In [8]:
from pathlib import Path

MODEL_NAME = "yolov8n.pt"

DATASET_YAML = "../datasets/data.yaml"

IMAGE_SIZE = 640

EPOCHS = 100

BATCH_SIZE = 16

OUTPUT_DIR = Path("../models").resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [9]:
MODEL_NAME = "yolov8n.pt"
DATASET_YAML = "../datasets/data.yaml"
IMAGE_SIZE = 640
EPOCHS = 100
BATCH_SIZE = 16
OUTPUT_DIR = Path("../models").resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# Verify Dataset

Confirm the fixed YOLOv8 dataset structure at `ai/datasets/` before training.


In [10]:
from pathlib import Path
import yaml

dataset_root = Path(DATASET_YAML).resolve().parent
data_yaml = Path(DATASET_YAML).resolve()

assert dataset_root.exists(), f'Dataset root {dataset_root} does not exist'
assert data_yaml.exists(), f'{DATASET_YAML} not found'

with open(data_yaml, 'r') as f:
    cfg = yaml.safe_load(f)

required_dirs = [
    dataset_root / 'train' / 'images',
    dataset_root / 'train' / 'labels',
    dataset_root / 'valid' / 'images',
    dataset_root / 'valid' / 'labels',
    dataset_root / 'test' / 'images',
    dataset_root / 'test' / 'labels',
]
for path in required_dirs:
    assert path.exists(), f'Required path not found: {path}'

train_count = sum(1 for _ in (dataset_root / 'train' / 'images').glob('*.jpg')) + sum(1 for _ in (dataset_root / 'train' / 'images').glob('*.jpeg')) + sum(1 for _ in (dataset_root / 'train' / 'images').glob('*.png'))
valid_count = sum(1 for _ in (dataset_root / 'valid' / 'images').glob('*.jpg')) + sum(1 for _ in (dataset_root / 'valid' / 'images').glob('*.jpeg')) + sum(1 for _ in (dataset_root / 'valid' / 'images').glob('*.png'))
test_count = sum(1 for _ in (dataset_root / 'test' / 'images').glob('*.jpg')) + sum(1 for _ in (dataset_root / 'test' / 'images').glob('*.jpeg')) + sum(1 for _ in (dataset_root / 'test' / 'images').glob('*.png'))

print('data.yaml exists:', data_yaml.exists())
print('train images:', train_count)
print('valid images:', valid_count)
print('test images:', test_count)
print('class names:', cfg.get('names') or cfg.get('class_names') or cfg.get('labels'))


data.yaml exists: True
train images: 255
valid images: 23
test images: 12
class names: ['license plate', 'wheel']


# Train YOLOv8

Train YOLOv8 Nano with the fixed dataset and specified training configuration.


In [11]:
from pathlib import Path
import torch
from ultralytics import YOLO

dataset_yaml = Path(DATASET_YAML).resolve()

device = 'mps' if getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

model = YOLO(MODEL_NAME)
model.train(
    data=str(dataset_yaml),
    imgsz=IMAGE_SIZE,
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    device=device,
    project=str(OUTPUT_DIR),
    name='train',
    exist_ok=True,
)
print('Training completed.')


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/Users/weiningyi/Library/Application Support/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Using device: mps
Ultralytics 8.4.91 🚀 Python-3.13.9 torch-2.13.0 MPS (Apple M1)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/weiningyi/Documents/smart-car-inspection/ai/datasets/data.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr

/opt/anaconda3/lib/python3.13/site-packages/torch/autograd/graph.py:979: UserWarning: index_put_with_accumulate_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/Context.cpp:190.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      1/100      4.25G      1.222      3.545       1.24         36        640: 100% ━━━━━━━━━━━━ 16/16 5.0s/it 1:200.8ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 4.5s/it 4.5s
                   all         23         38    0.00415      0.746      0.418      0.269

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


/opt/anaconda3/lib/python3.13/site-packages/torch/autograd/graph.py:979: UserWarning: index_put_with_accumulate_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/Context.cpp:190.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      2/100      4.29G     0.9563      1.806     0.9757         35        640: 100% ━━━━━━━━━━━━ 16/16 3.8s/it 1:013.1ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 4.8s/it 4.8s
                   all         23         38    0.00411      0.746      0.557      0.365

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


...

     98/100      4.25G     0.2888     0.2311     0.7991         21        640: 100% ━━━━━━━━━━━━ 16/16 3.9s/it 1:021.2sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 2.7s/it 2.7s
                   all         23         38      0.984      0.932      0.946      0.808

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     99/100      4.25G     0.2867     0.2313     0.7912         29        640: 0% ──────────── 0/16  4.4s

/opt/anaconda3/lib/python3.13/site-packages/torch/autograd/graph.py:979: UserWarning: index_put_with_accumulate_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/Context.cpp:190.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


     99/100      4.25G     0.3029     0.2374       0.79         27        640: 100% ━━━━━━━━━━━━ 16/16 6.4s/it 1:432.6sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 2.0s/it 2.0s
                   all         23         38      0.984      0.932      0.947      0.809

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
    100/100      4.25G     0.3289     0.2594     0.8196         29        640: 0% ──────────── 0/16  6.6s

/opt/anaconda3/lib/python3.13/site-packages/torch/autograd/graph.py:979: UserWarning: index_put_with_accumulate_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/Context.cpp:190.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


    100/100      4.25G     0.2866     0.2301     0.7831         25        640: 100% ━━━━━━━━━━━━ 16/16 3.7s/it 58.8s1.6s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 2.1s/it 2.1s
                   all         23         38      0.982      0.932      0.946      0.807

100 epochs completed in 2.080 hours.
Optimizer stripped from /Users/weiningyi/Documents/smart-car-inspection/ai/models/train/weights/last.pt, 6.3MB
Optimizer stripped from /Users/weiningyi/Documents/smart-car-inspection/ai/models/train/weights/best.pt, 6.3MB

Validating /Users/weiningyi/Documents/smart-car-inspection/ai/models/train/weights/best.pt...
Ultralytics 8.4.91 🚀 Python-3.13.9 torch-2.13.0 MPS (Apple M1)
Model summary (fused): 73 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.9s/it 1.9s
                   all         23         

# Validate Model

Run model validation on the trained weights to confirm the result.


In [12]:
from pathlib import Path
from ultralytics import YOLO

weights_path = OUTPUT_DIR / 'train' / 'weights' / 'best.pt'
assert weights_path.exists(), f'Expected trained best.pt not found at {weights_path}'

model = YOLO(str(weights_path))
results = model.val(data=str(DATASET_YAML), imgsz=IMAGE_SIZE, batch=BATCH_SIZE, device='cpu')
print('Validation completed.')
print('Validation results:', results.metrics if hasattr(results, 'metrics') else results)


Ultralytics 8.4.91 🚀 Python-3.13.9 torch-2.13.0 CPU (Apple M1)
Model summary (fused): 73 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.0±0.0 ms, read: 33.9±10.6 MB/s, size: 54.1 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
val: Scanning /Users/weiningyi/Documents/smart-car-inspection/ai/datasets/valid/labels.cache... 23 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 23/23 3.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 2.5s/it 5.1s<13.7s
                   all         23         38      0.951      0.931      0.976      0.824
         license plate         23         23      0.902          1      0.995      0.897
                 wheel         15         15          1      0.862      0.957      0.751
Speed: 1.9ms preprocess, 206.8ms inference, 0.0ms l

# Export best.pt

Copy the final `best.pt` file into `ai/models/best.pt` for standardized access.


In [13]:
from pathlib import Path
import shutil

weights_path = (OUTPUT_DIR / 'train' / 'weights' / 'best.pt').resolve()
assert weights_path.exists(), f'best.pt not found at {weights_path}'

best_destination = (OUTPUT_DIR / 'best.pt').resolve()
best_destination.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(weights_path, best_destination)

print('✓ best.pt exists')
print('✓ path', best_destination)
print('✓ model size (MB)', round(best_destination.stat().st_size / (1024 * 1024), 2))


✓ best.pt exists
✓ path /Users/weiningyi/Documents/smart-car-inspection/ai/models/best.pt
✓ model size (MB) 5.97
